In [2]:
!pip install scikeras

# Uninstall the current scikit-learn version
!pip uninstall scikit-learn -y

# Install a compatible version of scikit-learn (e.g., 1.4.2)
!pip install scikit-learn==1.4.2

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 59.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier

from sklearn.preprocessing import LabelEncoder, StandardScaler,OneHotEncoder


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    f1_score, fbeta_score,
    matthews_corrcoef, brier_score_loss
)

from collections import Counter
from imblearn.over_sampling import SMOTE

from sklearn.calibration import CalibrationDisplay
from sklearn.utils.class_weight import compute_class_weight

In [3]:
train_faults = pd.read_csv('imputed_training_faults_diagnostics.csv', low_memory=False)
train_faults.head()

,RecordID,EventTimeStamp,eventDescription,ecuSoftwareVersion,ecuModel,ecuMake,ecuSource,spn,fmi,active,...,FuelLtd,FuelRate,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,Speed,Throttle,TurboBoostPressure
0,1.0,2015-02-21 10:47:13,Low (Severity Low) Engine Coolant Level,unknown,unknown,unknown,0.0,111.0,17.0,True,...,-1.565238,-0.766937,1.431077,False,-1.048422,1023.0,True,-0.868869,-0.098845,-0.700382
1,2.0,2015-02-21 11:34:34,Low (Severity Low) Engine Coolant Level,unknown,unknown,unknown,11.0,629.0,12.0,True,...,-0.001624,-0.000864,0.000145,True,0.002972,1279.0,False,-0.003182,0.001744,-0.001398
2,3.0,2015-02-21 11:35:31,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11.0,1807.0,2.0,False,...,-0.001624,-0.000864,0.000145,True,0.002972,1279.0,False,-0.003182,0.001744,-0.001398
3,4.0,2015-02-21 11:35:33,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11.0,1807.0,2.0,True,...,-0.001624,-0.000864,0.000145,True,0.002972,1279.0,False,-0.003182,0.001744,-0.001398
4,5.0,2015-02-21 11:39:41,Low (Severity Low) Engine Coolant Level,22281684P01*22357957P01*22362082P01*,0USA13_13_0415_2238A,VOLVO,0.0,4364.0,17.0,False,...,-0.001624,-0.000864,0.000145,True,0.002972,16639.0,False,-0.003182,0.001744,-0.001398


In [4]:
# Convert timestamps to datetime objects
# train_faults['EventTimeStamp'] = pd.to_datetime(train_faults['EventTimeStamp'])
# train_faults['LocationTimeStamp'] = pd.to_datetime(train_faults['LocationTimeStamp'])

train_faults['EventTimeStamp'] = train_faults['EventTimeStamp'].astype('int64') // 10**9
train_faults['LocationTimeStamp'] = train_faults['LocationTimeStamp'].astype('int64') // 10**9

In [5]:
train_faults.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1058069 entries, 0 to 1058068
Data columns (total 46 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   RecordID                   1058069 non-null  float64
 1   EventTimeStamp             1058069 non-null  int64  
 2   eventDescription           1058069 non-null  object 
 3   ecuSoftwareVersion         1058069 non-null  object 
 4   ecuModel                   1058069 non-null  object 
 5   ecuMake                    1058069 non-null  object 
 6   ecuSource                  1058069 non-null  float64
 7   spn                        1058069 non-null  float64
 8   fmi                        1058069 non-null  float64
 9   active                     1058069 non-null  bool   
 10  activeTransitionCount      1058069 non-null  float64
 11  EquipmentID                1058069 non-null  object 
 12  MCTNumber                  1058069 non-null  float64
 13  Latitude    

In [6]:
features = train_faults.drop(['Derate_Target_12.0-0.001','Derate_Target_8.0-0.001','Derate_Target_4.0-0.001','Derate_Target_2.0-0.001','IsFullDerate'], axis = 1)

X = features
y = train_faults['Derate_Target_2.0-0.001']


In [7]:
train_faults['Derate_Target_2.0-0.001'].value_counts()

,count
Derate_Target_2.0-0.001,
0.0,1057396
1.0,673


Keras can not take strings directly , so converting string/categorical variables to
numerical


In [8]:
cat_cols = X.select_dtypes(include=["object", "bool"]).columns


In [9]:
ct = ColumnTransformer(
    transformers=[
        ('ohe',OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify = y, random_state = 321)

In [11]:
#Encode Categorical Columns
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

Next, we need to scale our predictors. We'll use a StandardScaler to transform the training and test data.

In [12]:
scaler = StandardScaler(with_mean= False)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Finally, we'll build a model, using the Sequential model class.



In [13]:
es = EarlyStopping(monitor='val_loss', patience=2)

In [14]:
# Function to create the Keras model for SciKeras
n_features = X_train.shape[1]
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(n_features,)))
    model.add(tf.keras.layers.Dense(128, activation='relu')) #relu for hidden layers, Relu outputs zero for negative values and keeps positive values unchanged
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid')) # sigmoid for binary classification, probabilty that engine will enter derate condition
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    epochs=100,   # 100 complete passthrough of training data
    batch_size=256,  #Keras split your training data into 2604 mini-batches.
    validation_split=0.1,
    callbacks=[es]
)


We are training a model to predict derate probability, using Adam to learn efficiently, binary crossentropy to measure error, and accuracy to evaluate performance.”


In [15]:
print(type(X_train))

<class 'scipy.sparse._csr.csr_matrix'>


In [16]:
# Create and fit model pipeline
pipe = Pipeline(
    steps=[
        #('scaler', StandardScaler(with_mean=False)), #only scales variance
        ('model', model)
    ]
).fit(X_train, y_train)


Epoch 1/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 62s 23ms/step - accuracy: 0.9993 - loss: 0.0046 - val_accuracy: 0.9995 - val_loss: 0.0025
Epoch 2/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 58s 22ms/step - accuracy: 0.9994 - loss: 0.0028 - val_accuracy: 0.9995 - val_loss: 0.0025
Epoch 3/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 83s 23ms/step - accuracy: 0.9994 - loss: 0.0028 - val_accuracy: 0.9995 - val_loss: 0.0025
Epoch 4/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 102s 30ms/step - accuracy: 0.9994 - loss: 0.0026 - val_accuracy: 0.9995 - val_loss: 0.0025
Epoch 5/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 63s 24ms/step - accuracy: 0.9994 - loss: 0.0025 - val_accuracy: 0.9995 - val_loss: 0.0025
Epoch 6/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 59s 22ms/step - accuracy: 0.9994 - loss: 0.0024 - val_accuracy: 0.9995 - val_loss: 0.0028
Epoch 7/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 61s 23ms/step - accuracy: 0.9994 - loss: 0.0024 - val_accuracy: 0.9995 - val_loss: 0.0027


In [22]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

2894/2894 ━━━━━━━━━━━━━━━━━━━━ 47s 16ms/step
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step


In [30]:
print(confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
))

[[107729 632448]
 [     1    470]]


Looks like the training data is highly imbalnced with 740,177(0) no derate and 471(1) derate.
740144 - TN (non-derate)
33 - FP (model wrongly said derate)
425 - FN (model missed actual derate events)
46 - TP (derate events)

In [26]:
print(confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)
)

[[ 46029 271190]
 [    10    192]]


Looks like the test data is highly imbalnced with 740,177(0) no derate and 471(1) derate.
317202 - TN (non-derate)
17 - FP (model wrongly said derate)
184 - FN (model missed actual derate events)
18 - TP (derate events)

In [27]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))
print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

         0.0       1.00      0.15      0.25    740177
         1.0       0.00      1.00      0.00       471

    accuracy                           0.15    740648
   macro avg       0.50      0.57      0.13    740648
weighted avg       1.00      0.15      0.25    740648

              precision    recall  f1-score   support

         0.0       1.00      0.15      0.25    317219
         1.0       0.00      0.95      0.00       202

    accuracy                           0.15    317421
   macro avg       0.50      0.55      0.13    317421
weighted avg       1.00      0.15      0.25    317421



The training recall is 0.10 model finds only 10% derate events
Test recall is 0.08 model finds only 9% derate events on the test data.
accuracy is 1.00 because almost all rows are class 0.

**fit the model by balancing class weights**

In [28]:

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print(class_weights)

{np.float64(0.0): np.float64(0.5003181671411028), np.float64(1.0): np.float64(786.2505307855627)}


In [29]:
pipe.fit(
    X_train,
    y_train,
    model__class_weight=class_weights
)

Epoch 1/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 82s 29ms/step - accuracy: 0.5050 - loss: 1.3023 - val_accuracy: 0.5764 - val_loss: 0.8106
Epoch 2/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 62s 24ms/step - accuracy: 0.8573 - loss: 0.4226 - val_accuracy: 0.6796 - val_loss: 0.8144


Pipeline(steps=[('model',
                 KerasClassifier(batch_size=256, callbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x7b3d54f66ed0>], epochs=100, model=<function create_model at 0x7b3d4eed6fc0>, validation_split=0.1))])

In [32]:
# Create and fit model pipeline
pipe .fit(X_train, y_train,model__class_weight={
        0: 1,
        1: 1000 #missing derate is 1000 time more costlier than missing a normal event
        }

      )


Epoch 1/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 65s 23ms/step - accuracy: 0.8722 - loss: 1.1120 - val_accuracy: 0.7324 - val_loss: 0.5113
Epoch 2/100
2604/2604 ━━━━━━━━━━━━━━━━━━━━ 59s 23ms/step - accuracy: 0.9473 - loss: 0.4035 - val_accuracy: 0.9715 - val_loss: 0.0916


Pipeline(steps=[('model',
                 KerasClassifier(batch_size=256, callbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x7b3d54f66ed0>], epochs=100, model=<function create_model at 0x7b3d4eed6fc0>, validation_split=0.1))])

In [33]:
confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
)

array([[107729, 632448],
       [     1,    470]])

In [35]:
cm = confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)


TN, FP, FN, TP = cm.ravel()

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print(f'Money saved:{TP * 4000 - FP * 500}')

True Negatives (TN): 46029
False Positives (FP): 271190
False Negatives (FN): 10
True Positives (TP): 192
Money saved:-134827000


After using class weights , still the training data set is imbalanced

 **Adjust the Threshold**

First, we'll split into a train and validation set so we can adjust various hyperparameters, and then a test set, so that we can see how well our model performs on unseen data.

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify = y, random_state = 321, train_size = 0.8)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, stratify = y_train, random_state = 321, train_size = 0.6/0.8)

In [39]:
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

In [40]:
scaler = StandardScaler(with_mean= False)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [41]:
X_val = ct.transform(X_val)
X_val = scaler.transform(X_val)

In [46]:
n_features = X_train.shape[1]
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(n_features,)))
    model.add(tf.keras.layers.Dense(128, activation='relu')) #relu for hidden layers, Relu outputs zero for negative values and keeps positive values unchanged
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid')) # sigmoid for binary classification, probabilty that engine will enter derate condition
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    epochs=100,   # 100 complete passthrough of training data
    batch_size=256,  #Keras split your training data into 2604 mini-batches.
    validation_split=0.1,
    callbacks=[es]
)

In [47]:
model.fit(X_train, y_train, validation_data=(X_val, y_val))

Epoch 1/100
2480/2480 ━━━━━━━━━━━━━━━━━━━━ 67s 25ms/step - accuracy: 0.9994 - loss: 0.0045 - val_accuracy: 0.9994 - val_loss: 0.0030
Epoch 2/100
2480/2480 ━━━━━━━━━━━━━━━━━━━━ 66s 27ms/step - accuracy: 0.9994 - loss: 0.0029 - val_accuracy: 0.9994 - val_loss: 0.0028


KerasClassifier(
	model=<function create_model at 0x7b3c66435da0>
	build_fn=None
	warm_start=False
	random_state=None
	optimizer=rmsprop
	loss=None
	metrics=None
	batch_size=256
	validation_batch_size=None
	verbose=1
	callbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x7b3d54f66ed0>]
	validation_split=0.1
	shuffle=True
	run_eagerly=False
	epochs=100
	class_weight=None
)

In [49]:
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(f'MCC: {matthews_corrcoef(y_test, y_pred)}')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

827/827 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step
Accuracy: 0.9993809483304508
MCC: 0.17207930442450323
[[211479      0]
 [   131      4]]
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    211479
         1.0       1.00      0.03      0.06       135

    accuracy                           1.00    211614
   macro avg       1.00      0.51      0.53    211614
weighted avg       1.00      1.00      1.00    211614



In [50]:
y_val_pred_proba = model.predict_proba(X_val)[:,1]
y_val_pred_proba

827/827 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step


array([2.3803352e-05, 3.4366549e-05, 3.4374483e-05, ..., 7.0985302e-06,
       3.9691886e-06, 1.1003324e-05], dtype=float32)

In [51]:
candidate_thresholds = np.arange(start = 0.1, stop = 0.925, step = 0.01)
thresholds = pd.DataFrame({'threshold': candidate_thresholds})
thresholds['f1'] = thresholds['threshold'].apply(lambda x: f1_score(y_val, y_val_pred_proba > x))
thresholds.sort_values('f1', ascending = False).head()

,threshold,f1
9,0.19,0.313869
15,0.25,0.308370
5,0.15,0.305630
10,0.20,0.303030
6,0.16,0.301994


In [55]:
threshold = 0.19
y_pred_proba = model.predict_proba(X_test)[:,1]

y_pred = y_pred_proba > threshold
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(f'MCC: {matthews_corrcoef(y_test, y_pred)}')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))



827/827 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step
Accuracy: 0.9991777481641101
MCC: 0.3503446733084609
[[211393     86]
 [    88     47]]
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    211479
         1.0       0.35      0.35      0.35       135

    accuracy                           1.00    211614
   macro avg       0.68      0.67      0.68    211614
weighted avg       1.00      1.00      1.00    211614



TypeError: missing a required argument: 'y_pred'

In [56]:
cm = confusion_matrix(y_test, y_pred)

TN, FP, FN, TP = cm.ravel()

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print(f'Money saved:{TP * 4000 - FP * 500}')

True Negatives (TN): 211393
False Positives (FP): 86
False Negatives (FN): 88
True Positives (TP): 47
Money saved:145000


Looks like model is producing zero probabilities for each sample and looks like threshold tuning is ineffective.

**Now let's see how model perform on unseen data**

In [ ]:
test_faults = pd.read_csv('testing_faults_diagnostics.csv', low_memory=False)
test_faults.head()

,RecordID,EventTimeStamp,eventDescription,ecuSoftwareVersion,ecuModel,ecuMake,ecuSource,spn,fmi,active,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1100564,2019-01-01 00:24:10,Low (Severity Medium) Engine Coolant Level,PC4__1284P4C_6*,MX16U13D13,PCAR,0,111,18,True,...,60.8,True,59.0,17407,True,NaN,0.00000,NaN,0.0,0.58
1,1100565,2019-01-01 00:36:08,Low (Severity Medium) Engine Coolant Level,PC4__1284P4C_6*,MX16U13D13,PCAR,0,111,18,False,...,NaN,NaN,NaN,17407,NaN,NaN,NaN,NaN,NaN,NaN
2,1100566,2019-01-01 01:54:35,Low (Severity Medium) Engine Coolant Level,05317106*04150360*061416163421*09401361*G1*BDR*,6X1u13D1500000000,CMMNS,0,111,18,True,...,NaN,True,87.8,18431,True,NaN,0.00000,NaN,100.0,0.00
3,1100567,2019-01-01 02:08:21,Low (Severity Medium) Engine Coolant Level,05317106*04150360*061416163421*09401361*G1*BDR*,6X1u13D1500000000,CMMNS,0,111,18,False,...,NaN,NaN,NaN,17407,NaN,NaN,NaN,NaN,NaN,NaN
4,1100568,2019-01-01 03:06:52,Not Reporting Data Front Operator Wiper Switch,NaN,NaN,NaN,49,2863,7,True,...,NaN,True,105.8,255,False,NaN,66.89449,NaN,100.0,17.11


In [ ]:
test_faults.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129266 entries, 0 to 129265
Data columns (total 48 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   RecordID                   129266 non-null  int64  
 1   EventTimeStamp             129266 non-null  object 
 2   eventDescription           124155 non-null  object 
 3   ecuSoftwareVersion         59792 non-null   object 
 4   ecuModel                   120111 non-null  object 
 5   ecuMake                    120111 non-null  object 
 6   ecuSource                  129266 non-null  int64  
 7   spn                        129266 non-null  int64  
 8   fmi                        129266 non-null  int64  
 9   active                     129266 non-null  bool   
 10  activeTransitionCount      129266 non-null  int64  
 11  EquipmentID                129266 non-null  int64  
 12  MCTNumber                  129266 non-null  int64  
 13  Latitude                   12

In [ ]:
test_faults = test_faults.drop(['Derate_Target_12.0-0.001','Derate_Target_8.0-0.001','Derate_Target_4.0-0.001','IsFullDerate'], axis = 1)


In [ ]:
test_faults = test_faults.fillna(X.median(numeric_only=True))

In [ ]:
cat_cols = test_faults.select_dtypes(include=["object", "bool"]).columns



In [ ]:
pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('transformer', ct),
        ('model', model)
    ]
).fit(X_train, y_train)

In [ ]:
y_pred = pipe.predict(test_faults[features])

ValueError: Boolean array expected for the condition, not float64